# PyKIS 테스트 노트북

한국투자증권 OpenAPI를 활용한 PyKIS 라이브러리 테스트 노트북입니다.

## 환경 설정

먼저 `.env` 파일에 다음 정보가 설정되어 있어야 합니다:
```
KIS_APP_KEY=your_app_key
KIS_APP_SECRET=your_app_secret
KIS_ACCOUNT_NO=your_account_number
KIS_ACCOUNT_CODE=01
KIS_BASE_URL=https://openapi.koreainvestment.com:9443
```

In [1]:
# 필요한 라이브러리 임포트
import os
from dotenv import load_dotenv
from pykis import Agent
import pandas as pd
from pprint import pprint

# 환경 변수 로드
load_dotenv()

print("✅ 라이브러리 임포트 완료")

✅ 라이브러리 임포트 완료


## 1. Agent 초기화

In [2]:
# Agent 인스턴스 생성 - 키 값들을 직접 매핑(수동입력)
agent = Agent(
    app_key="PSNm3H5Yq9FVeRr0GjVOgz5r85KvlbVg60EE",
    app_secret="+ri3poGokecM6GbhhR2vnO6KNHaX/EHg0Qa6+k4urU7rKBQbEHX9Zt7bU5atCvEb3kXT3ESrjm0OGm9TpWIo5ANsGcmoH7O0wV0PqmV+U4aB09M8h48oQbRjsExnnx6xGPYtWYttopwNbiwGq5M7X9oWRvTyvh6LGTbz91kRnOrn6SMoBPU=",
    account_no="68867843",
    account_code="01",
)
print("✅ Agent 초기화 완료")
print(f"계좌번호: {os.getenv('KIS_ACCOUNT_NO')}")

2025-11-18 08:36:22,165 - pykis.core.client - INFO - 토큰 발급을 시작합니다...
2025-11-18 08:36:22,166 - pykis.core.client - INFO - 토큰 발급 완료 (만료: 2025-11-18 08:36:22)


[Agent] 토큰이 없거나 만료되었습니다. 새 토큰을 발급받습니다.
[Agent] 토큰 발급이 완료되었습니다.
✅ Agent 초기화 완료
계좌번호: 68867843


## 2. 주식 현재가 조회

In [5]:
# 삼성전자(005930) 현재가 조회
stock_code = "005930"
price_data = agent.get_stock_price(stock_code)

if price_data and price_data.get("rt_cd") == "0":
    output = price_data["output"]
    print(f"\n📊 {stock_code} (삼성전자) 현재가 정보")
    print("=" * 50)
    print(f"현재가: {output.get('stck_prpr'):>10}원")
    print(f"전일대비: {output.get('prdy_vrss'):>8}원 ({output.get('prdy_ctrt')}%)")
    print(f"시가: {output.get('stck_oprc'):>12}원")
    print(f"고가: {output.get('stck_hgpr'):>12}원")
    print(f"저가: {output.get('stck_lwpr'):>12}원")
    print(f"거래량: {output.get('acml_vol'):>10}주")
    print(f"거래대금: {int(output.get('acml_tr_pbmn', 0)):>8}백만원")
else:
    print("❌ 현재가 조회 실패")
    pprint(price_data)


itemchart = agent.inquire_daily_itemchartprice(
    stock_code, 
    start_date='20250101', 
    end_date='20250131',
    period="D"
)
print(itemchart)


📊 005930 (삼성전자) 현재가 정보
현재가:     100600원
전일대비:        0원 (0.00%)
시가:            0원
고가:            0원
저가:            0원
거래량:         11주
거래대금:  1106600백만원
{'output1': {'prdy_vrss': '0', 'prdy_vrss_sign': '3', 'prdy_ctrt': '0.00', 'stck_prdy_clpr': '100600', 'acml_vol': '11', 'acml_tr_pbmn': '1106600', 'hts_kor_isnm': '삼성전자', 'stck_prpr': '100600', 'stck_shrn_iscd': '005930', 'prdy_vol': '17192584', 'stck_mxpr': '130700', 'stck_llam': '70500', 'stck_oprc': '0', 'stck_hgpr': '0', 'stck_lwpr': '0', 'stck_prdy_oprc': '99800', 'stck_prdy_hgpr': '101000', 'stck_prdy_lwpr': '99500', 'askp': '0', 'bidp': '0', 'prdy_vrss_vol': '-17192573', 'vol_tnrt': '0.00', 'stck_fcam': '100', 'lstn_stcn': '5919637922', 'cpfn': '7780', 'hts_avls': '5955156', 'per': '20.32', 'eps': '4950.00', 'pbr': '1.74', 'itewhol_loan_rmnd_ratem name': '0.28'}, 'output2': [{'stck_bsop_date': '20250131', 'stck_clpr': '52400', 'stck_oprc': '52200', 'stck_hgpr': '53000', 'stck_lwpr': '51700', 'acml_vol': '42186279', 'acml_tr_pb

## 3. 주식 호가 조회

In [4]:
# 호가 정보 조회
orderbook = agent.get_orderbook(stock_code)

if orderbook and orderbook.get("rt_cd") == "0":
    output = orderbook["output"]
    print(f"\n📋 {stock_code} 호가 정보")
    print("=" * 60)
    print(f"{'매도호가':>12} | {'수량':>10} || {'수량':>10} | {'매수호가':>12}")
    print("-" * 60)
    
    for i in range(1, 11):
        ask_price = output.get(f'askp{i}', '0')
        ask_vol = output.get(f'askp_rsqn{i}', '0')
        bid_price = output.get(f'bidp{i}', '0')
        bid_vol = output.get(f'bidp_rsqn{i}', '0')
        print(f"{ask_price:>12} | {ask_vol:>10} || {bid_vol:>10} | {bid_price:>12}")
else:
    print("❌ 호가 조회 실패")
    pprint(orderbook)

2025-11-18 08:36:22,364 - pykis.core.client - INFO - 토큰 발급을 시작합니다...
2025-11-18 08:36:22,365 - pykis.core.client - INFO - 토큰 발급 완료 (만료: 2025-11-18 08:36:22)


KeyError: 'output'

## 4. 주식현재가 일자별 조회 (최근 30건)

In [ ]:
# 최근 30일 일봉 데이터 조회
daily_price = agent.inquire_daily_price(stock_code, period="D")

if daily_price and daily_price.get("rt_cd") == "0":
    df = pd.DataFrame(daily_price["output"])
    
    # 컬럼명 한글로 변경
    df_display = df[['stck_bsop_date', 'stck_oprc', 'stck_hgpr', 'stck_lwpr', 'stck_clpr', 'acml_vol']].copy()
    df_display.columns = ['일자', '시가', '고가', '저가', '종가', '거래량']
    
    print(f"\n📈 {stock_code} 최근 30일 일봉 데이터")
    print("=" * 80)
    print(df_display.head(10))
    print(f"\n총 {len(df)}건의 데이터")
else:
    print("❌ 일자별 조회 실패")
    pprint(daily_price)

## 5. 국내주식 기간별 시세 조회 (날짜 범위 지정)

In [ ]:
# 특정 기간 조회 (2024년 1월 ~ 2024년 3월)
start_date = "20240101"
end_date = "20240331"

itemchart = agent.inquire_daily_itemchartprice(
    stock_code, 
    start_date=start_date, 
    end_date=end_date,
    period="D"
)

if itemchart and itemchart.get("rt_cd") == "0":
    df = pd.DataFrame(itemchart["output1"])
    
    # 컬럼명 한글로 변경
    df_display = df[['stck_bsop_date', 'stck_oprc', 'stck_hgpr', 'stck_lwpr', 'stck_clpr', 'acml_vol']].copy()
    df_display.columns = ['일자', '시가', '고가', '저가', '종가', '거래량']
    
    print(f"\n📊 {stock_code} 기간별 시세 ({start_date} ~ {end_date})")
    print("=" * 80)
    print(df_display.head(10))
    print(f"\n총 {len(df)}건의 데이터")
    
    # 요약 정보
    if "output2" in itemchart:
        output2 = itemchart["output2"]
        print("\n📋 기간 요약 정보")
        print(f"전일대비: {output2.get('prdy_vrss', 'N/A')}원 ({output2.get('prdy_ctrt', 'N/A')}%)")
else:
    print("❌ 기간별 조회 실패")
    pprint(itemchart)

## 6. 분봉 데이터 조회

In [ ]:
# 당일 분봉 데이터 조회 (최근 120분)
minute_data = agent.get_minute_price(stock_code, hour="153000")

if minute_data and minute_data.get("rt_cd") == "0":
    df = pd.DataFrame(minute_data["output2"])
    
    # 컬럼명 한글로 변경
    df_display = df[['stck_bsop_date', 'stck_cntg_hour', 'stck_prpr', 'stck_oprc', 'stck_hgpr', 'stck_lwpr', 'cntg_vol']].copy()
    df_display.columns = ['일자', '시각', '현재가', '시가', '고가', '저가', '체결량']
    
    print(f"\n⏰ {stock_code} 분봉 데이터 (최근 120분)")
    print("=" * 100)
    print(df_display.head(20))
    print(f"\n총 {len(df)}건의 분봉 데이터")
else:
    print("❌ 분봉 조회 실패")
    pprint(minute_data)

## 7. 계좌 정보 조회

In [ ]:
# 계좌 잔고 조회
balance = agent.get_account_balance()

if balance and balance.get("rt_cd") == "0":
    output1 = balance.get("output1", [])
    output2 = balance.get("output2", {})
    
    print("\n💰 계좌 잔고 정보")
    print("=" * 80)
    
    # 보유 종목 리스트
    if output1:
        df = pd.DataFrame(output1)
        df_display = df[['pdno', 'prdt_name', 'hldg_qty', 'pchs_avg_pric', 'prpr', 'evlu_pfls_amt']].copy()
        df_display.columns = ['종목코드', '종목명', '보유수량', '매입평균가', '현재가', '평가손익']
        print("\n📋 보유 종목")
        print(df_display)
    
    # 계좌 요약
    print("\n📊 계좌 요약")
    print(f"예수금: {output2.get('dnca_tot_amt', 'N/A'):>15}원")
    print(f"총평가금액: {output2.get('tot_evlu_amt', 'N/A'):>11}원")
    print(f"평가손익: {output2.get('evlu_pfls_smtl_amt', 'N/A'):>13}원")
    print(f"수익률: {output2.get('evlu_pfls_rt', 'N/A'):>15}%")
else:
    print("❌ 계좌 조회 실패")
    pprint(balance)

## 8. 투자자별 매매 동향

In [ ]:
# 투자자별 매매 동향 조회
investor = agent.get_stock_investor(stock_code)

if investor and investor.get("rt_cd") == "0":
    output = investor.get("output", [])
    
    if output:
        df = pd.DataFrame(output)
        print(f"\n👥 {stock_code} 투자자별 매매 동향")
        print("=" * 80)
        print(df[['stck_bsop_date', 'stck_prpr', 'prsn_ntby_qty', 'frgn_ntby_qty', 'orgn_ntby_qty']].head(10))
        print("\n컬럼: 일자, 종가, 개인순매수, 외국인순매수, 기관순매수")
else:
    print("❌ 투자자 동향 조회 실패")
    pprint(investor)

## 9. 상위 상승주 조회

In [ ]:
# 상위 상승주 조회 (KOSPI)
top_gainers = agent.get_top_gainers(market="KOSPI")

if top_gainers and top_gainers.get("rt_cd") == "0":
    output = top_gainers.get("output", [])
    
    if output:
        df = pd.DataFrame(output)
        print("\n🚀 KOSPI 상위 상승주 TOP 20")
        print("=" * 80)
        print(df[['data_rank', 'hts_kor_isnm', 'stck_prpr', 'prdy_vrss', 'prdy_ctrt', 'acml_vol']].head(20))
        print("\n컬럼: 순위, 종목명, 현재가, 전일대비, 등락률, 거래량")
else:
    print("❌ 상승주 조회 실패")
    pprint(top_gainers)

## 10. 프로그램 매매 동향

In [ ]:
# 종목별 프로그램 매매 동향
program_trade = agent.get_program_trade_by_stock(stock_code)

if program_trade and program_trade.get("rt_cd") == "0":
    output = program_trade.get("output", [])
    
    if output:
        df = pd.DataFrame(output)
        print(f"\n🤖 {stock_code} 프로그램 매매 동향")
        print("=" * 80)
        print(df.head(10))
else:
    print("❌ 프로그램 매매 조회 실패")
    pprint(program_trade)

## 11. 여러 종목 비교 분석

In [ ]:
# 여러 종목 현재가 비교
stocks = {
    "005930": "삼성전자",
    "000660": "SK하이닉스",
    "035420": "NAVER",
    "035720": "카카오",
    "005380": "현대차"
}

comparison_data = []

for code, name in stocks.items():
    price_data = agent.get_stock_price(code)
    if price_data and price_data.get("rt_cd") == "0":
        output = price_data["output"]
        comparison_data.append({
            "종목코드": code,
            "종목명": name,
            "현재가": output.get("stck_prpr"),
            "전일대비": output.get("prdy_vrss"),
            "등락률": output.get("prdy_ctrt") + "%",
            "거래량": output.get("acml_vol"),
            "거래대금": output.get("acml_tr_pbmn")
        })

if comparison_data:
    df_comparison = pd.DataFrame(comparison_data)
    print("\n📊 주요 종목 비교")
    print("=" * 100)
    print(df_comparison)
else:
    print("❌ 종목 비교 실패")

## 12. 캔들스틱 차트 시각화 (선택사항)

In [ ]:
# matplotlib이 설치되어 있다면 차트 그리기
try:
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates
    from datetime import datetime
    
    # 일봉 데이터로 차트 그리기
    if daily_price and daily_price.get("rt_cd") == "0":
        df_chart = pd.DataFrame(daily_price["output"])
        df_chart['date'] = pd.to_datetime(df_chart['stck_bsop_date'])
        df_chart['stck_clpr'] = df_chart['stck_clpr'].astype(int)
        
        plt.figure(figsize=(14, 6))
        plt.plot(df_chart['date'], df_chart['stck_clpr'], marker='o', linewidth=2)
        plt.title(f'{stock_code} 최근 30일 종가 추이', fontsize=16, fontweight='bold')
        plt.xlabel('날짜', fontsize=12)
        plt.ylabel('종가 (원)', fontsize=12)
        plt.grid(True, alpha=0.3)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
        
        print("✅ 차트 생성 완료")
except ImportError:
    print("ℹ️ matplotlib이 설치되어 있지 않습니다. 차트 시각화를 건너뜁니다.")
    print("설치: pip install matplotlib")

## 13. API 메서드 탐색

Agent가 제공하는 모든 메서드를 검색할 수 있습니다.

In [ ]:
# 'price' 키워드로 메서드 검색
price_methods = agent.search_methods("price")

print("\n🔍 'price' 관련 메서드")
print("=" * 80)
for method in price_methods[:10]:
    print(f"- {method['name']}: {method['description'][:50]}...")

In [ ]:
# 특정 메서드의 사용법 확인
agent.show_method_usage("inquire_daily_itemchartprice")

## 마무리

이 노트북에서 다룬 주요 기능:

1. ✅ 주식 현재가 조회
2. ✅ 호가 정보 조회
3. ✅ 일자별 시세 조회 (최근 30건)
4. ✅ 기간별 시세 조회 (날짜 범위 지정)
5. ✅ 분봉 데이터 조회
6. ✅ 계좌 정보 조회
7. ✅ 투자자별 매매 동향
8. ✅ 상위 상승주 조회
9. ✅ 프로그램 매매 동향
10. ✅ 여러 종목 비교
11. ✅ 차트 시각화
12. ✅ API 메서드 탐색

### 더 알아보기

- [PyKIS GitHub](https://github.com/unohee/pykis)
- [한국투자증권 OpenAPI 문서](https://apiportal.koreainvestment.com)
- [PyKIS 문서](https://pykis.readthedocs.io)